# GraphRAG Notebook Driver
This notebook uses a LangGraph workflow (`graphrag/graphrag_langgraph.py`) for hybrid Cypher + vector retrieval over Neo4j.

In [ ]:
# Run once in this venv if needed:
# %pip install langgraph langchain langchain-openai neo4j numpy pandas python-dotenv

In [1]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ").strip()

print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

OPENAI_API_KEY set: True


In [2]:
from graphrag_helpers import GraphRAGAssistant

NEO4J_ENV_FILE = "Neo4j-d2f72502-Created-2026-03-01.txt"

rag = GraphRAGAssistant.from_neo4j_env_file(
    NEO4J_ENV_FILE,
    model="gpt-4o-mini",
    embeddings_model="text-embedding-3-small",
)
print("GraphRAG assistant initialized.")

GraphRAG assistant initialized.


In [3]:
rag.build_vector_index(force_rebuild=False)
print("Hybrid vector index ready.")

Hybrid vector index ready.


In [4]:
def ask_graphrag(
    question: str,
    max_retries: int = 2,
    interactive_clarification: bool = True,
    clarification_text: str | None = None,
):
    callback = None
    if clarification_text:
        used = {"done": False}
        def _one_shot_callback(clarification_question: str, prior_result: dict):
            if not used["done"]:
                used["done"] = True
                return clarification_text
            return None
        callback = _one_shot_callback
        interactive_clarification = True
    if interactive_clarification:
        result = rag.ask_with_user_clarification(
            question,
            max_retries=max_retries,
            top_n_default=5,
            k_vector=6,
            max_clarification_rounds=2,
            clarification_callback=callback,
        )
    else:
        result = rag.ask(question, max_retries=max_retries, top_n_default=5, k_vector=6)
    print(result["answer"])
    return result


In [ ]:
# Example:
# ask_graphrag("For Singapore in 2024, show top import/export partners and explain whether it behaves like a shadow hub.")

In [5]:
ask_graphrag("For Singapore in 2024, show top import/export partners and explain whether it behaves like a shadow hub.")

c:\Users\stile\Desktop\Graph RAG\venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QuestionPlan(intent='hub_...nd shadow hub behavior'), input_type=QuestionPlan])
  return self.__pydantic_serializer__.to_python(


In 2024, Singapore's top export partners are Indonesia, Malaysia, Australia, China, and Vietnam, with trade values of approximately $10.99 billion, $10.76 billion, $6.84 billion, $3.73 billion, and $2.61 billion, respectively [C1]. For imports, the leading partners are India, Malaysia, Brazil, the USA, and China, with trade values of about $12.88 billion, $12.87 billion, $10.68 billion, $9.20 billion, and $7.85 billion, respectively [C2].

Singapore exhibits characteristics of a shadow hub, as indicated by its shadow residual of 0.055, which suggests it is unusually central in the oil trade network relative to its trade volume [V1]. Its shadow rank is 4, and it has a betweenness score of 0.109, indicating that Singapore plays a significant intermediary role in global trade flows [V1]. The high trade total of approximately $150.89 billion further supports this notion, as it reflects Singapore's extensive involvement in international trade [V1].

Overall, Singapore's trade dynamics and m

c:\Users\stile\Desktop\Graph RAG\venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QualityReport(is_sufficie...ne, retrieval_gap=False), input_type=QualityReport])
  return self.__pydantic_serializer__.to_python(


{'answer': "In 2024, Singapore's top export partners are Indonesia, Malaysia, Australia, China, and Vietnam, with trade values of approximately $10.99 billion, $10.76 billion, $6.84 billion, $3.73 billion, and $2.61 billion, respectively [C1]. For imports, the leading partners are India, Malaysia, Brazil, the USA, and China, with trade values of about $12.88 billion, $12.87 billion, $10.68 billion, $9.20 billion, and $7.85 billion, respectively [C2].\n\nSingapore exhibits characteristics of a shadow hub, as indicated by its shadow residual of 0.055, which suggests it is unusually central in the oil trade network relative to its trade volume [V1]. Its shadow rank is 4, and it has a betweenness score of 0.109, indicating that Singapore plays a significant intermediary role in global trade flows [V1]. The high trade total of approximately $150.89 billion further supports this notion, as it reflects Singapore's extensive involvement in international trade [V1].\n\nOverall, Singapore's trad

In [14]:
ask_graphrag("Across all years, which country is the top shadow hub?")

c:\Users\stile\Desktop\Graph RAG\venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QualityReport(is_sufficie...ne, retrieval_gap=False), input_type=QualityReport])
  return self.__pydantic_serializer__.to_python(
c:\Users\stile\Desktop\Graph RAG\venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RewritePlan(refined_quest...alysis of shadow hubs.'), input_type=RewritePlan])
  return self.__pydantic_serializer__.to_python(
c:\Users\stile\Desktop\Graph RAG\venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed'

The top shadow hub across all available years in the dataset is the United States (USA). In each year from 2020 to 2024, the USA consistently held the highest shadow rank, indicating its central role in the oil trade network. For instance, in 2024, the USA had a shadow residual of 0.224, a shadow rank of 1.0, and a trade total of approximately $826 billion, reflecting its significant position in the trade network [C1][V2][V3][V5].

Citations:
- [C1] Top shadow hubs by residual | Neo4j SHADOW_HUB relationship | year=2024, rows=5
- [V1] Vector hit shadow::USA::2024 | shadow_metric | score=0.627
- [V2] Vector hit shadow::USA::2023 | shadow_metric | score=0.620
- [V3] Vector hit shadow::USA::2022 | shadow_metric | score=0.619
- [V4] Vector hit shadow::HKG::2024 | shadow_metric | score=0.617
- [V5] Vector hit shadow::USA::2020 | shadow_metric | score=0.616
- [V6] Vector hit shadow::GBR::2024 | shadow_metric | score=0.614


{'answer': 'The top shadow hub across all available years in the dataset is the United States (USA). In each year from 2020 to 2024, the USA consistently held the highest shadow rank, indicating its central role in the oil trade network. For instance, in 2024, the USA had a shadow residual of 0.224, a shadow rank of 1.0, and a trade total of approximately $826 billion, reflecting its significant position in the trade network [C1][V2][V3][V5].\n\nCitations:\n- [C1] Top shadow hubs by residual | Neo4j SHADOW_HUB relationship | year=2024, rows=5\n- [V1] Vector hit shadow::USA::2024 | shadow_metric | score=0.627\n- [V2] Vector hit shadow::USA::2023 | shadow_metric | score=0.620\n- [V3] Vector hit shadow::USA::2022 | shadow_metric | score=0.619\n- [V4] Vector hit shadow::HKG::2024 | shadow_metric | score=0.617\n- [V5] Vector hit shadow::USA::2020 | shadow_metric | score=0.616\n- [V6] Vector hit shadow::GBR::2024 | shadow_metric | score=0.614',
 'plan': {'intent': 'top_shadow_hubs',
  'count

In [ ]:
ask_graphrag("Give me a list of sanctioned countries that also are shadow hubs and have been growing in that activity over time")

In [7]:
ask_graphrag("Give me the country with the greatest shadow residual over all years")

c:\Users\stile\Desktop\Graph RAG\venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QuestionPlan(intent='top_...eval_focus='shadow_hub'), input_type=QuestionPlan])
  return self.__pydantic_serializer__.to_python(


The country with the greatest shadow residual over all years is the United States (USA). In 2024, it has a shadow residual of 0.2243, ranking it first among the countries analyzed. This indicates that the USA is unusually central in the oil trade network relative to its trade volume [C1]. Additionally, in 2023, the USA had an even higher shadow residual of 0.3052, further solidifying its position as the top shadow hub [V2].

Citations:
- [C1] Top shadow hubs by residual | Neo4j SHADOW_HUB relationship | year=2024, rows=5
- [V1] Vector hit shadow::USA::2024 | shadow_metric | score=0.643
- [V2] Vector hit shadow::USA::2023 | shadow_metric | score=0.639
- [V3] Vector hit shadow::CAN::2024 | shadow_metric | score=0.635
- [V4] Vector hit shadow::GBR::2024 | shadow_metric | score=0.633
- [V5] Vector hit shadow::USA::2022 | shadow_metric | score=0.632
- [V6] Vector hit shadow::CHN::2024 | shadow_metric | score=0.630


c:\Users\stile\Desktop\Graph RAG\venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QualityReport(is_sufficie...ne, retrieval_gap=False), input_type=QualityReport])
  return self.__pydantic_serializer__.to_python(


{'answer': 'The country with the greatest shadow residual over all years is the United States (USA). In 2024, it has a shadow residual of 0.2243, ranking it first among the countries analyzed. This indicates that the USA is unusually central in the oil trade network relative to its trade volume [C1]. Additionally, in 2023, the USA had an even higher shadow residual of 0.3052, further solidifying its position as the top shadow hub [V2].\n\nCitations:\n- [C1] Top shadow hubs by residual | Neo4j SHADOW_HUB relationship | year=2024, rows=5\n- [V1] Vector hit shadow::USA::2024 | shadow_metric | score=0.643\n- [V2] Vector hit shadow::USA::2023 | shadow_metric | score=0.639\n- [V3] Vector hit shadow::CAN::2024 | shadow_metric | score=0.635\n- [V4] Vector hit shadow::GBR::2024 | shadow_metric | score=0.633\n- [V5] Vector hit shadow::USA::2022 | shadow_metric | score=0.632\n- [V6] Vector hit shadow::CHN::2024 | shadow_metric | score=0.630',
 'plan': {'intent': 'top_shadow_hubs',
  'country_name

In [ ]:
# Close when finished:
rag.close()